In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("processed_skincare.csv")

In [5]:
df.columns

Index(['product_name', 'brand_name', 'ingredients', 'rating', 'reviews',
       'price_usd', 'highlights', 'primary_category', 'secondary_category',
       'harmful_found', 'clean_score', 'clean_label', 'detected_irritants',
       'irritant_count'],
      dtype='object')

In [6]:
df = df.dropna(subset=["rating"])

In [7]:
df["recommendation_score"] = (
    df["clean_score"] * 0.7 +
    df["rating"] * 20 * 0.3
)

In [8]:
df[[
    "product_name",
    "clean_score",
    "rating",
    "recommendation_score"
]].head()

,product_name,clean_score,rating,recommendation_score
0,GENIUS Sleeping Collagen Moisturizer,85,4.5413,86.7478
1,GENIUS Liquid Collagen Serum,75,4.0259,76.6554
2,Triple Algae Eye Renewal Balm Eye Cream,90,4.5306,90.1836
3,GENIUS Liquid Collagen Lip Treatment,90,3.8721,86.2326
4,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,65,4.4134,71.9804


In [9]:
recommend_products("Moisturizers")

NameError: name 'recommend_products' is not defined

In [10]:
recommend_products("Sunscreen")

NameError: name 'recommend_products' is not defined

In [ ]:
recommend_products("Eye Care")

In [11]:
recommend_sensitive_skin("Moisturizers")

NameError: name 'recommend_sensitive_skin' is not defined

In [12]:
def irritant_level(count):

    if count == 0:
        return "Very Low"

    elif count <= 2:
        return "Low"

    elif count <= 4:
        return "Moderate"

    else:
        return "High"

In [13]:
df["irritant_level"] = df["irritant_count"].apply(irritant_level)

In [14]:
fragrance_words = [
    "fragrance",
    "parfum",
    "essential oil",
    "limonene",
    "linalool",
    "citral"
]

In [15]:
def is_fragrance_free(ingredients):

    ingredients = str(ingredients).lower()

    for word in fragrance_words:
        if word in ingredients:
            return "No"

    return "Yes"

In [16]:
df["fragrance_free"] = df["ingredients"].apply(is_fragrance_free)

In [17]:
comedogenic_ingredients = [
    "coconut oil",
    "isopropyl myristate",
    "algae extract",
    "cocoa butter",
    "lanolin"
]

In [18]:
def acne_safe(ingredients):

    ingredients = str(ingredients).lower()

    for ingredient in comedogenic_ingredients:
        if ingredient in ingredients:
            return "No"

    return "Yes"

In [19]:
df["acne_prone_safe"] = df["ingredients"].apply(acne_safe)

In [20]:
def detect_skin_type(ingredients):

    ingredients = str(ingredients).lower()

    if "salicylic acid" in ingredients:
        return "Oily / Acne-Prone"

    elif "hyaluronic acid" in ingredients:
        return "Dry Skin"

    elif "ceramide" in ingredients:
        return "Sensitive Skin"

    else:
        return "All Skin Types"

In [21]:
df["skin_type_match"] = df["ingredients"].apply(detect_skin_type)

In [22]:
def best_for(ingredients):

    ingredients = str(ingredients).lower()

    if "retinol" in ingredients:
        return "Anti-Aging"

    elif "niacinamide" in ingredients:
        return "Oil Control & Brightening"

    elif "salicylic acid" in ingredients:
        return "Acne"

    elif "hyaluronic acid" in ingredients:
        return "Dryness"

    else:
        return "General Care"

In [23]:
df["best_for"] = df["ingredients"].apply(best_for)

In [24]:
df[[
    "product_name",
    "clean_score",
    "irritant_level",
    "fragrance_free",
    "acne_prone_safe",
    "skin_type_match",
    "best_for"
]].head(10)

,product_name,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for
0,GENIUS Sleeping Collagen Moisturizer,85,Moderate,No,Yes,Sensitive Skin,General Care
1,GENIUS Liquid Collagen Serum,75,High,No,Yes,All Skin Types,Oil Control & Brightening
2,Triple Algae Eye Renewal Balm Eye Cream,90,Low,Yes,Yes,Sensitive Skin,General Care
3,GENIUS Liquid Collagen Lip Treatment,90,Low,Yes,Yes,All Skin Types,General Care
4,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,65,High,No,Yes,All Skin Types,General Care
5,GENIUS Ultimate Anti-Aging Cream,80,Moderate,No,Yes,Sensitive Skin,General Care
6,GENIUS Ultimate Anti-Aging Melting Cleanser,85,Moderate,No,Yes,All Skin Types,General Care
7,Gentle Rejuvenating Cleanser,90,Low,Yes,Yes,All Skin Types,General Care
8,10 Day Results Kit,75,High,No,Yes,Sensitive Skin,Oil Control & Brightening
9,Advanced Anti-Aging Repairing Oil,95,Low,Yes,Yes,Sensitive Skin,General Care


In [25]:
def recommend_products(category, top_n=5):

    filtered = df[
        df["secondary_category"]
        .str.contains(category, case=False, na=False)
    ]

    sorted_products = filtered.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return sorted_products[[
        "product_name",
        "brand_name",
        "rating",
        "clean_score",
        "irritant_level",
        "fragrance_free",
        "acne_prone_safe",
        "skin_type_match",
        "best_for"
    ]].head(top_n)

In [40]:
recommend_products("Moisturizers")

,product_name,brand_name,rating,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for
2149,Barrier Culture Moisturizer with Niacinamide &...,The Nue Co.,5.0000,100,Very Low,Yes,Yes,Sensitive Skin,Oil Control & Brightening
755,Soothe + Hydrate Barrier Building Hydrosol Mist,Flora + Bast,5.0000,100,Very Low,Yes,Yes,All Skin Types,General Care
392,Phyto Nature Oxygen Cream,Dermalogica,4.8646,100,Very Low,Yes,Yes,All Skin Types,General Care
351,Rice Drops Face Oil Hydrating Serum,DAMDAM,4.8548,100,Very Low,Yes,Yes,All Skin Types,General Care
1644,Evercalm Barrier Support Face Oil,REN Clean Skincare,4.8391,100,Very Low,Yes,Yes,All Skin Types,General Care


In [28]:
df.to_csv("recommendation_ready_skincare.csv", index=False)

In [29]:
def personalized_recommendation(
    skin_type=None,
    concern=None,
    fragrance_free=None,
    acne_safe=None,
    top_n=5
):
    
    filtered = df.copy()

    # Skin type filtering
    if skin_type:
        filtered = filtered[
            filtered["skin_type_match"]
            .str.contains(skin_type, case=False, na=False)
        ]

    # Concern filtering
    if concern:
        filtered = filtered[
            filtered["best_for"]
            .str.contains(concern, case=False, na=False)
        ]

    # Fragrance-free filtering
    if fragrance_free == "Yes":
        filtered = filtered[
            filtered["fragrance_free"] == "Yes"
        ]

    # Acne-safe filtering
    if acne_safe == "Yes":
        filtered = filtered[
            filtered["acne_prone_safe"] == "Yes"
        ]

    # Sort by recommendation score
    filtered = filtered.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return filtered[[
        "product_name",
        "brand_name",
        "clean_score",
        "irritant_level",
        "fragrance_free",
        "acne_prone_safe",
        "skin_type_match",
        "best_for",
        "rating"
    ]].head(top_n)

In [43]:
personalized_recommendation(
    concern="Acne"
)

,product_name,brand_name,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,rating
64,The Skin Renewal System,Augustinus Bader,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,5.0000
1748,Acne Treatment Gel,SEPHORA COLLECTION,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,5.0000
1747,Clarifying Peel Pads Purify + Exfoliate,SEPHORA COLLECTION,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,5.0000
43,Juneberry & Collagen Hydrating Cold Cream Clea...,alpyn beauty,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,4.8950
1699,Resurfacing Peel Mask,SEPHORA COLLECTION,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,4.8299


In [44]:
personalized_recommendation(
    skin_type="Sensitive"
)

,product_name,brand_name,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,rating
2149,Barrier Culture Moisturizer with Niacinamide &...,The Nue Co.,100,Very Low,Yes,Yes,Sensitive Skin,Oil Control & Brightening,5.0000
1284,Plumping High Performance Lip Filler with Hyal...,MACRENE actives,100,Very Low,Yes,Yes,Sensitive Skin,General Care,4.7692
693,1% Vitamin A Retinol Serum,Farmacy,100,Very Low,Yes,Yes,Sensitive Skin,Anti-Aging,4.7660
1932,Dream Oasis Deep Hydration Serum,Summer Fridays,100,Very Low,Yes,Yes,Sensitive Skin,Oil Control & Brightening,4.7361
696,Honey Halo Moisturizer Jumbo,Farmacy,100,Very Low,Yes,Yes,Sensitive Skin,General Care,4.7273


In [45]:
personalized_recommendation(
    skin_type="Dry",
    fragrance_free="Yes"
)

,product_name,brand_name,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,rating
2150,"Barrier Culture Cleanser Pre-, Pro- & Postbiot...",The Nue Co.,100,Very Low,Yes,Yes,Dry Skin,Dryness,5.0000
2148,Skin Filter Daily Brightening Phyto-Retinol + ...,The Nue Co.,100,Very Low,Yes,Yes,Dry Skin,Dryness,5.0000
690,Wake Up Honey Eye Cream with Brightening Vitam...,Farmacy,100,Very Low,Yes,Yes,Dry Skin,Dryness,4.7118
917,Super Nova 5% THD Vitamin C + Caffeine Brighte...,Herbivore,100,Very Low,Yes,Yes,Dry Skin,Dryness,4.6207
156,Premier Cru Dark Circle Correcting Eye Cream,Caudalie,100,Very Low,Yes,Yes,Dry Skin,Dryness,4.6139


In [46]:
personalized_recommendation(
    acne_safe="Yes"
)

,product_name,brand_name,clean_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,rating
64,The Skin Renewal System,Augustinus Bader,100,Very Low,Yes,Yes,Oily / Acne-Prone,Acne,5.0
924,Aquarius BHA + Blue Tansy Clarity Cleanser,Herbivore,100,Very Low,Yes,Yes,All Skin Types,General Care,5.0
1361,SuperPower Immune Support Supplement,Moon Juice,100,Very Low,Yes,Yes,All Skin Types,General Care,5.0
755,Soothe + Hydrate Barrier Building Hydrosol Mist,Flora + Bast,100,Very Low,Yes,Yes,All Skin Types,General Care,5.0
2058,Superkind Refining Cleanser,Tata Harper,100,Very Low,Yes,Yes,All Skin Types,General Care,5.0


In [47]:
pd.set_option("display.max_colwidth", 50)

In [48]:
def explain_recommendation(row):

    explanation = f"""
    Recommended because:
    
    - Clean Score: {row['clean_score']}
    - Irritant Level: {row['irritant_level']}
    - Skin Match: {row['skin_type_match']}
    - Best For: {row['best_for']}
    - Fragrance-Free: {row['fragrance_free']}
    - Acne-Prone Safe: {row['acne_prone_safe']}
    """

    return explanation

In [49]:
sample = personalized_recommendation(
    concern="Acne"
)

sample.iloc[0]

product_name       The Skin Renewal System
brand_name                Augustinus Bader
clean_score                            100
irritant_level                    Very Low
fragrance_free                         Yes
acne_prone_safe                        Yes
skin_type_match          Oily / Acne-Prone
best_for                              Acne
rating                                 5.0
Name: 64, dtype: object

In [50]:
sample = personalized_recommendation(
    concern="Acne"
)

sample.iloc[0]

product_name       The Skin Renewal System
brand_name                Augustinus Bader
clean_score                            100
irritant_level                    Very Low
fragrance_free                         Yes
acne_prone_safe                        Yes
skin_type_match          Oily / Acne-Prone
best_for                              Acne
rating                                 5.0
Name: 64, dtype: object

In [51]:
explain_recommendation(sample.iloc[0])

'\n    Recommended because:\n    \n    - Clean Score: 100\n    - Irritant Level: Very Low\n    - Skin Match: Oily / Acne-Prone\n    - Best For: Acne\n    - Fragrance-Free: Yes\n    - Acne-Prone Safe: Yes\n    '

In [52]:
df.to_csv("clean_beauty_final.csv", index=False)

In [53]:
import os

os.getcwd()

'C:\\Users\\fj18y\\skincare-recommender\\Untitled Folder\\Untitled Folder'

In [54]:
os.listdir()

['.ipynb_checkpoints',
 'cleaned_skincare.csv',
 'clean_beauty_final.csv',
 'dataset_exploration.ipynb',
 'ingredient_analysis.ipynb',
 'processed_skincare.csv',
 'product_info.csv',
 'recommendation_ready_skincare.csv',
 'recommendation_system.ipynb',
 'reviews_0-250.csv',
 'reviews_1250-end.csv',
 'reviews_500-750.csv',
 'reviews_750-1250.csv',
 'Untitled.ipynb',
 'Untitled1.ipynb',
 'Untitled2.ipynb',
 'Untitled3.ipynb',
 'Untitled4.ipynb',
 'Untitled5.ipynb']

In [55]:
df["secondary_category"].value_counts()


secondary_category
Moisturizers              539
Treatments                459
Cleansers                 354
Value & Gift Sets         190
Eye Care                  184
Masks                     161
Mini Size                 108
Sunscreen                 105
Lip Balms & Treatments     60
Wellness                   51
Self Tanners               48
High Tech Tools            26
Shop by Concern             1
Name: count, dtype: int64

In [58]:
df = pd.read_csv("clean_beauty_final.csv")
print(df.columns)

Index(['product_name', 'brand_name', 'ingredients', 'rating', 'reviews',
       'price_usd', 'highlights', 'primary_category', 'secondary_category',
       'harmful_found', 'clean_score', 'clean_label', 'detected_irritants',
       'irritant_count', 'recommendation_score', 'irritant_level',
       'fragrance_free', 'acne_prone_safe', 'skin_type_match', 'best_for'],
      dtype='object')


In [59]:
for col in df.columns:
    print(col)

product_name
brand_name
ingredients
rating
reviews
price_usd
highlights
primary_category
secondary_category
harmful_found
clean_score
clean_label
detected_irritants
irritant_count
recommendation_score
irritant_level
fragrance_free
acne_prone_safe
skin_type_match
best_for


In [60]:
"clean_score" in df.columns

True

In [30]:
df = pd.read_csv(
    r"C:\Users\fj18y\skincare-recommender\Untitled Folder\Untitled Folder\clean_beauty_final.csv"
)

In [35]:
df.sort_values(
    by=[
        "clean_score",
        "irritant_level",
        "recommendation_score",
        "rating",
        "reviews"
    ],
    ascending=[
        False,
        True,
        False,
        False,
        False
    ]
)

,product_name,brand_name,ingredients,rating,reviews,price_usd,highlights,primary_category,secondary_category,harmful_found,...,clean_label,detected_irritants,irritant_count,recommendation_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,irritant_rank
924,Aquarius BHA + Blue Tansy Clarity Cleanser,Herbivore,"['aqua/water/eau, sodium lauroyl methyl isethi...",5.000000,15.0,26.0,"['Vegan', 'Clean + Planet Positive', 'Good for...",Skincare,Cleansers,[],...,Clean Beauty,[],0,100.000000,Very Low,Yes,Yes,All Skin Types,General Care,NaN
2148,Skin Filter Daily Brightening Phyto-Retinol + ...,The Nue Co.,"['aqua (water), glycerin, isoamyl laurate, hep...",5.000000,12.0,65.0,"['Vegan', 'Good for: Dullness/Uneven Texture',...",Skincare,Treatments,[],...,Clean Beauty,[],0,100.000000,Very Low,Yes,Yes,Dry Skin,Dryness,NaN
2149,Barrier Culture Moisturizer with Niacinamide &...,The Nue Co.,"['aqua (water), dicaprylyl carbonate, glycerin...",5.000000,7.0,65.0,"['Vegan', 'Clean + Planet Positive', 'Hydratin...",Skincare,Moisturizers,[],...,Clean Beauty,[],0,100.000000,Very Low,Yes,Yes,Sensitive Skin,Oil Control & Brightening,NaN
2150,"Barrier Culture Cleanser Pre-, Pro- & Postbiot...",The Nue Co.,"['aqua (water), glycerin, sodium lauroyl sarco...",5.000000,6.0,42.0,"['Vegan', 'Clean + Planet Positive', 'Hydratin...",Skincare,Cleansers,[],...,Clean Beauty,[],0,100.000000,Very Low,Yes,Yes,Dry Skin,Dryness,NaN
755,Soothe + Hydrate Barrier Building Hydrosol Mist,Flora + Bast,"['cannabis hydrosol, lavender hydrosol, cucumb...",5.000000,3.0,44.0,"['Best for Dry Skin', 'Hydrating', 'Good for: ...",Skincare,Moisturizers,[],...,Clean Beauty,[],0,100.000000,Very Low,Yes,Yes,All Skin Types,General Care,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1766,Benefiance Wrinkle Smoothing Retinol Serum,Shiseido,"['water(aqua/eau),butylene glycol,pentaerythri...",4.580600,937.0,75.0,"['Retinol', 'Good for: Anti-Aging', 'Without P...",Skincare,Treatments,"['methylparaben', 'ethylparaben', 'alcohol den...",...,Contains Potential Irritants,"['methylparaben', 'ethylparaben', 'alcohol den...",11,58.983600,High,No,Yes,All Skin Types,General Care,3.0
1807,Wrinkle Correcting Skincare Set,Shiseido,['shiseido benefiance wrinkle smoothing eye cr...,4.232174,NaN,64.0,"['Best for Normal Skin', 'Best for Combination...",Skincare,Value & Gift Sets,"['methylparaben', 'ethylparaben', 'alcohol den...",...,Contains Potential Irritants,"['methylparaben', 'ethylparaben', 'alcohol den...",11,56.893045,High,No,Yes,All Skin Types,General Care,3.0
1305,Acne Facial Cleanser,Mario Badescu,"['water, glycerin, triethanolamine, sodium lau...",3.762500,80.0,15.0,NaN,Skincare,Cleansers,"['methylparaben', 'propylparaben', 'ethylparab...",...,Contains Potential Irritants,"['methylparaben', 'propylparaben', 'ethylparab...",11,54.075000,High,No,Yes,Oily / Acne-Prone,Acne,3.0
1325,Anti-Aging Regimen Kit,Mario Badescu,"['glycolic foaming cleanser:', 'aqua (water, e...",4.300000,10.0,30.0,NaN,Skincare,Value & Gift Sets,"['methylparaben', 'propylparaben', 'ethylparab...",...,Contains Potential Irritants,"['methylparaben', 'propylparaben', 'ethylparab...",12,53.800000,High,No,No,All Skin Types,General Care,3.0


In [32]:
df["irritant_level"].value_counts()

irritant_level
Low         946
Moderate    462
High        444
Very Low    434
Name: count, dtype: int64

In [36]:
mapping = {
    "Very Low": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4
}

df["irritant_rank"] = df["irritant_level"].map(mapping)

In [38]:
df["irritant_rank"].isnull().sum()

np.int64(0)

In [39]:
df[["irritant_level", "irritant_rank"]].drop_duplicates()

,irritant_level,irritant_rank
0,Moderate,3
1,High,4
2,Low,2
27,Very Low,1


In [40]:
filtered = df.copy()

filtered = filtered.sort_values(
    by=[
        "clean_score",
        "irritant_rank",
        "recommendation_score",
        "rating",
        "reviews"
    ],
    ascending=[False, True, False, False, False]
)

filtered.head()

,product_name,brand_name,ingredients,rating,reviews,price_usd,highlights,primary_category,secondary_category,harmful_found,...,clean_label,detected_irritants,irritant_count,recommendation_score,irritant_level,fragrance_free,acne_prone_safe,skin_type_match,best_for,irritant_rank
924,Aquarius BHA + Blue Tansy Clarity Cleanser,Herbivore,"['aqua/water/eau, sodium lauroyl methyl isethi...",5.0,15.0,26.0,"['Vegan', 'Clean + Planet Positive', 'Good for...",Skincare,Cleansers,[],...,Clean Beauty,[],0,100.0,Very Low,Yes,Yes,All Skin Types,General Care,1
2148,Skin Filter Daily Brightening Phyto-Retinol + ...,The Nue Co.,"['aqua (water), glycerin, isoamyl laurate, hep...",5.0,12.0,65.0,"['Vegan', 'Good for: Dullness/Uneven Texture',...",Skincare,Treatments,[],...,Clean Beauty,[],0,100.0,Very Low,Yes,Yes,Dry Skin,Dryness,1
2149,Barrier Culture Moisturizer with Niacinamide &...,The Nue Co.,"['aqua (water), dicaprylyl carbonate, glycerin...",5.0,7.0,65.0,"['Vegan', 'Clean + Planet Positive', 'Hydratin...",Skincare,Moisturizers,[],...,Clean Beauty,[],0,100.0,Very Low,Yes,Yes,Sensitive Skin,Oil Control & Brightening,1
2150,"Barrier Culture Cleanser Pre-, Pro- & Postbiot...",The Nue Co.,"['aqua (water), glycerin, sodium lauroyl sarco...",5.0,6.0,42.0,"['Vegan', 'Clean + Planet Positive', 'Hydratin...",Skincare,Cleansers,[],...,Clean Beauty,[],0,100.0,Very Low,Yes,Yes,Dry Skin,Dryness,1
755,Soothe + Hydrate Barrier Building Hydrosol Mist,Flora + Bast,"['cannabis hydrosol, lavender hydrosol, cucumb...",5.0,3.0,44.0,"['Best for Dry Skin', 'Hydrating', 'Good for: ...",Skincare,Moisturizers,[],...,Clean Beauty,[],0,100.0,Very Low,Yes,Yes,All Skin Types,General Care,1
